In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

class InfoNCE(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_q, z_k):
        # zq, zk shape: (batch_size, dim)
        sim = z_q @ z_k.T / self.temperature    # (batch_size, batch_size)
        pos_sim = torch.diag(sim).unsqueeze(1)  # (batch_size, 1)

        exp_sim = torch.exp(sim)    # (batch_size, batch_size)
        denominator = exp_sim.sum(dim=1, keepdim=True)  # (batch_size, 1)
        # log(a/b) = loga - logb，其中pos_sim就是log(exp(sim+))
        log_prob = pos_sim - torch.log(denominator)     # (batch_size, 1)
        loss = -log_prob.mean()
        return loss